# Laya on `laya_v3`: run 6's recipe on the final envelopes (Kaggle, GPU T4 ×2)

**Not run yet: you decide when to retrain.** Same recipe as run 6 (`laya_run6_kaggle.ipynb`: `v3_single` and `v3_mixed`,
verdict only, 2 epochs, RLCD, max_len 2048), one change: the rows.

Run 6 read `laya_v2` and rebuilt every row in the notebook (`to_v3_state` and its own verdict question). The app builds
the same envelope with the engine, and the two differ (rewrite phase 2's skew report: every hadith and dhikr row, the
question on every row, and the passages the engine now groups differently). Since rewrite phase 22 the dataset stores
**the final envelopes** (ADR 0003): `laya_v3` is built by `server/wahy/laya/laya_v3.py` with the engine's own v3
builder, the same call the app makes at verification time, and this notebook reads the rows **as they are**. Nothing
here reshapes a state or a question.

| run | data | envelope | questions | what it answers |
|---|---|---|---|---|
| `v3_single` | `laya_v3/v3_single` | v3, as the app builds it | verdict only | single verses, served as trained |
| `v3_mixed` | `laya_v3/v3_mixed` | v3, as the app builds it | verdict only | single verses + passages, served as trained |

The rows keep the envelope's three questions (verdict, start_missing, end_missing); `--questions verdict` trains the
verdict item only, whose input is the stored state and question unchanged (the app filters a checkpoint's questions
the same way).

**Benchmarks:** run 6 `v3_mixed` (in the app today), `before: soft_asis` (91.4% accuracy, 0.944 AUC, 6.4% real rejected).

### Kaggle settings (right-hand panel)
* **Accelerator:** `GPU T4 x2` · **Internet:** `On` · **Secrets:** `HF_TOKEN`
* **Input:** the `laya_v3` dataset (upload `data/laya_datagen/laya_v3.zip`: `uv run python -m wahy.laya zip ...`)
* **Run:** *Save Version → Save & Run All (Commit)*. Expect **~25–30 min** total.
* **Download afterwards:** `results/` folder.


## 1. Environment & dual T4 GPU check

In [ ]:
!nvidia-smi
import os, subprocess, torch

n_gpu = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | Visible GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")

assert n_gpu >= 2, (
    f"Expected 2 GPUs, but detected {n_gpu}! Notebook options -> Accelerator -> GPU T4 x2."
)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## 2. Install dependencies

In [ ]:
# pinned to the 0.4 line the Wahy data was validated with (laya.train / laya.calibrate are identical 0.4.12 -> 0.4.15)
!pip install -q "laya>=0.4.12,<0.5" transformers torch packaging accelerate

## 3. Dataset, scripts, base checkpoint, Hub login

In [ ]:
import glob, json, sys
from pathlib import Path
from huggingface_hub import snapshot_download, login
from laya.agent import _fix_tokenizer_config

DATA = next(Path(p).parent for p in sorted(glob.glob("/kaggle/input/**/manifest.json", recursive=True))
            if json.loads(Path(p).read_text()).get("name") == "laya_v3")
manifest = json.loads((DATA / "manifest.json").read_text())
sys.path.insert(0, str(DATA))
import eval_laya

print("dataset:", DATA, "| created", manifest.get("created"), "| base model", manifest.get("base_model"))

WORK = Path("/kaggle/working")
for d in ("results", "logs", "data"):
    (WORK / d).mkdir(parents=True, exist_ok=True)

# The rows are final (ADR 0003): read them as they are, never reshape them here.
assert manifest.get("rows_are_final"), "this notebook reads a dataset of final envelopes (laya_v3)"
V3_SINGLE_DIR = DATA / "v3_single"
V3_MIXED_DIR = DATA / "v3_mixed"
for d in (V3_SINGLE_DIR, V3_MIXED_DIR):
    for split in ("train", "val", "test"):
        print(f"  {d.name} {split}: {sum(1 for _ in open(d / f'{split}.jsonl'))} rows")

MODEL_ID = manifest["base_model"]
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)
print("base checkpoint:", model_dir)

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("Hub: logged in")
except Exception as e:
    HF_TOKEN = None
    print("Hub: no HF_TOKEN secret:", e)


## 4. Settings
Both models train on **Verdict Only** with 2 epochs, effective batch 64 (4 × 8 accumulation), max_len 2048, and RLCD loss.

In [ ]:
RUNS = {name: {"data_dir": DATA / name,
               "questions": manifest["variants"][name]["cfg_extra"]["wahy_questions"],
               "cfg_extra": manifest["variants"][name]["cfg_extra"],      # envelope 3 and its options, from the dataset
               "about": manifest["variants"][name]["about"]}
        for name in ("v3_single", "v3_mixed")}

TRAIN_ARGS = (f"--epochs 2 --micro-batch 4 --grad-accum 8 --lr-encoder 2.5e-5 --lr-head 1e-4 --loss rlcd "
              f"--max-len 2048 --head-max-len 256")
APP_FALSE_SHARE = 0.10
BEFORE_MODEL = "ZenoAbderrahim/laya-wahy-soft_asis"
PUSH_TO_HUB = True
RUN_TAG = "v3data"                 # name the run before you start it (run 7?)
HUB_PREFIX = f"ZenoAbderrahim/laya-wahy-{RUN_TAG}"

def run(cmd, log):
    print("$", cmd, flush=True)
    with open(log, "w") as f:
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            f.write(line)
            if "Warning" not in line and "warn(" not in line:
                print(line, end="", flush=True)
        return p.wait()

import pandas as pd
rows = []
for name, cfg in RUNS.items():
    tr_lines = [json.loads(l) for l in open(cfg["data_dir"] / "train.jsonl")]
    n_rows = len(tr_lines)
    n_false = sum(r["meta"]["verdict"] == "FALSE" for r in tr_lines)
    rows.append({
        "run": name,
        "questions": ", ".join(cfg["questions"]),
        "about": cfg["about"],
        "train rows": n_rows,
        "not a quote (share)": round(n_false / n_rows, 3)
    })
pd.DataFrame(rows).set_index("run")


## 5. The "before" line: current production model

In [ ]:
flags = f"--verse-flags {DATA}/test_verse_flags.json"
if HF_TOKEN:
    rc = run(f"python {DATA}/eval_laya.py predict --model {BEFORE_MODEL} --name 'before: soft_asis (app)' "
             f"--test {DATA}/v1_test.jsonl --out {WORK}/results/before_soft_asis.json --app-false-share {APP_FALSE_SHARE} "
             f"--batch-size 32 --baselines {DATA}/v1_baselines.json {flags}",
             WORK / "logs" / "eval_before.log")
    print("before:", "ok" if rc == 0 else f"failed (exit {rc}), see logs/eval_before.log")

## 6. Train + evaluate

In [ ]:
import time
status = {}

for name, cfg in RUNS.items():
    data_dir = cfg["data_dir"]
    qs = cfg["questions"]
    out = WORK / "models" / name
    t0 = time.time()
    try:
        cfg_extra_clean = json.dumps(cfg["cfg_extra"]).replace('"', '\\"')
        qarg = f"--questions {','.join(qs)}" if qs else ""
        rc = run(f"torchrun --standalone --nproc_per_node=2 {DATA}/train_ddp.py {model_dir} {out} "
                 f"--train {data_dir}/train.jsonl --val {data_dir}/val.jsonl --name laya-wahy-{RUN_TAG}-{name} "
                 f"--cfg-extra \"{cfg_extra_clean}\" {qarg} {TRAIN_ARGS}", WORK / "logs" / f"train_{name}.log")
        if rc != 0 or not (out / "model.safetensors").exists():
            status[name] = f"training failed (exit {rc}), see logs/train_{name}.log"
            continue
        
        tr_lines = [json.loads(l) for l in open(data_dir / "train.jsonl")]
        tr_false_share = round(sum(r["meta"]["verdict"] == "FALSE" for r in tr_lines) / len(tr_lines), 4)
        
        rc = run(f"python {DATA}/eval_laya.py predict --model {out} --name {name} --test {data_dir}/test.jsonl "
                 f"--out {WORK}/results/{name}.json --train-false-share {tr_false_share} "
                 f"--app-false-share {APP_FALSE_SHARE} --batch-size 16 {flags}", WORK / "logs" / f"eval_{name}.log")
        status[name] = f"ok in {(time.time() - t0) / 60:.0f} min" if rc == 0 else f"evaluation failed (exit {rc})"
        train_log = out / "train_log.json"
        if train_log.exists():
            (WORK / "results" / f"train_log_{name}.json").write_text(train_log.read_text())
    except Exception as e:
        status[name] = f"error: {e}"
    print(f"\n##### {name}: {status[name]}\n", flush=True)

json.dump(status, open(WORK / "results" / "status.json", "w"), indent=1)
status

## 7. Load results and benchmarks

In [ ]:
from IPython.display import Markdown, display

v1_rows = eval_laya.read_jsonl(DATA / "v1_test.jsonl")
vflags = json.loads((DATA / "test_verse_flags.json").read_text())
res = {n: json.load(open(WORK / "results" / f"{n}.json")) for n in RUNS if (WORK / "results" / f"{n}.json").exists()}
before = [json.load(open(WORK / "results" / "before_soft_asis.json"))] if (WORK / "results" / "before_soft_asis.json").exists() else []
teachers = [json.load(open(f)) for f in sorted(glob.glob(str(WORK / "results" / "baseline_*.json")))]
models = before + [res[n] for n in RUNS if n in res]

print("Models loaded:", [m["name"] for m in models])

## 8. Verdict evaluation (Overall & Context Slices)

In [ ]:
report = []
for g in ["you", "you + Sonnet", "Sonnet · other lectures", "written text"]:
    t = f"### verdict · {g}\n" + eval_laya.compare(models + teachers, g)
    report.append(t); display(Markdown(t))

for g in ["you", "you + Sonnet"]:
    for part in ("by context", "by span length"):
        t = f"### verdict {part} · {g}\n" + eval_laya.compare_slices(models, g, part)
        report.append(t); display(Markdown(t))

(WORK / "results" / "verdict_report.md").write_text("\n\n".join(report), encoding="utf-8")

## 9. Decision rule & comparison against benchmarks

In [ ]:
print("=== BENCHMARK COMPARISON (Cards: you) ===")
print("Run 1 (soft_asis production):  AUC 0.944 | Acc 91.4% | Real rejected 6.4%")
print("Run 2 (curve_100):             AUC 0.945 | Acc 89.4% | Real rejected 9.9%")
print("Run 4 (v1_verdict):            AUC 0.933 | Acc 91.4% | Real rejected 4.1%")
print("Run 5 (v1_passages):           AUC 0.924 | Acc 84.9% | Real rejected 14.0%")
print("========================================\n")

target_model = "v3_mixed" if "v3_mixed" in res else list(res.keys())[0]
if target_model in res:
    r_you = res[target_model]["groups"]["you"]["verdict"]
    auc_val = r_you["AUC P(not a quote)"]
    acc_val = 100.0 * r_you["accuracy"][0]
    rej_val = 100.0 * r_you["real quotes wrongly rejected"][0]
    print(f"{target_model} result on 'you': AUC {auc_val:.3f} | Acc {acc_val:.1f}% | Real rejected {rej_val:.1f}%")
    if auc_val >= 0.94 and acc_val >= 90.0 and rej_val <= 7.0:
        print(f"\nSUCCESS: {target_model} beat/matched benchmarks with low false rejection!")
        print("-> Ready to deploy in production!")
    else:
        print(f"\nNotice: AUC is {auc_val:.3f}, Acc is {acc_val:.1f}%, Real rejected {rej_val:.1f}%.")


## 10. Visual comparison

In [ ]:
import matplotlib.pyplot as plt

names = [m["name"] for m in models]
aucs = [m["groups"].get("you", {}).get("verdict", {}).get("AUC P(not a quote)", 0) for m in models]
accs = [(100.0 * m["groups"].get("you", {}).get("verdict", {}).get("accuracy", [0])[0]) for m in models]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.bar(names, aucs, color="skyblue")
ax1.axhline(0.945, color="red", linestyle="--", label="Run 2 benchmark (0.945)")
ax1.set_title("Verdict AUC (Cards: you)")
ax1.set_ylim(0.75, 1.0)
ax1.legend()

ax2.bar(names, accs, color="salmon")
ax2.axhline(91.4, color="red", linestyle="--", label="soft_asis (91.4%)")
ax2.set_title("Verdict Accuracy % (Cards: you)")
ax2.set_ylim(70, 100)
ax2.legend()

plt.tight_layout()
plt.savefig(WORK / "results" / f"comparison_{RUN_TAG}.png")
plt.show()


## 11. (Optional) Push models to Hugging Face Hub (Private)

In [ ]:
if PUSH_TO_HUB and HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    for n in RUNS:
        out = WORK / "models" / n
        if not (out / "model.safetensors").exists():
            continue
        repo_id = f"{HUB_PREFIX}-{n}"
        api.create_repo(repo_id=repo_id, private=True, exist_ok=True)
        api.upload_folder(folder_path=str(out), repo_id=repo_id)
        print(f"pushed {repo_id}")